In [ ]:
# ── H10 Cell 1: All Dependencies + BSDTGravityEngine ─────────────────
import torch, numpy as np, time
from numba import njit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))

# ── SAT checker ───────────────────────────────────────────────────────
def check_sat(x, clauses):
    n_sat = 0
    for (vs, ss) in clauses:
        if any((x[v].item() == 1) == (s == 1) for v, s in zip(vs, ss)):
            n_sat += 1
    return n_sat == len(clauses), n_sat

# ── Numba WalkSAT (flat-array adjacency, no lists-of-lists) ──────────
@njit(cache=True)
def walksat_numba(clauses_arr, assignment, max_flips=100000, p=0.4):
    m = clauses_arr.shape[0]
    k = 3
    n = assignment.shape[0]

    # Count how many clauses each variable appears in
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(k):
            v = abs(clauses_arr[c, j]) - 1
            var_count[v] += 1

    # Build flat adjacency: var_offset[v] .. var_offset[v]+var_count[v]
    var_offset = np.zeros(n + 1, dtype=np.int32)
    for v in range(n):
        var_offset[v + 1] = var_offset[v] + var_count[v]
    total_entries = var_offset[n]
    var_adj = np.zeros(total_entries, dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(k):
            v = abs(clauses_arr[c, j]) - 1
            var_adj[var_offset[v] + fill[v]] = c
            fill[v] += 1

    # Init clause satisfaction counts
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        cnt = 0
        for j in range(k):
            v = abs(clauses_arr[c, j]) - 1
            s = 1 if clauses_arr[c, j] > 0 else 0
            if assignment[v] == s:
                cnt += 1
        clause_sat[c] = cnt

    for flip in range(max_flips):
        # Collect unsatisfied clauses
        n_unsat = 0
        for c in range(m):
            if clause_sat[c] == 0:
                n_unsat += 1
        if n_unsat == 0:
            return assignment, flip
        # Pick random unsatisfied clause
        target = np.random.randint(n_unsat)
        ci = -1
        cnt = 0
        for c in range(m):
            if clause_sat[c] == 0:
                if cnt == target:
                    ci = c
                    break
                cnt += 1
        # Random walk or greedy min-break
        if np.random.random() < p:
            j = np.random.randint(k)
            v = abs(clauses_arr[ci, j]) - 1
        else:
            best_v = abs(clauses_arr[ci, 0]) - 1
            best_break = 999999
            for j in range(k):
                v_cand = abs(clauses_arr[ci, j]) - 1
                brk = 0
                for idx in range(var_offset[v_cand], var_offset[v_cand + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 1:
                        # Check if v_cand is the sole satisfier
                        for jj in range(k):
                            vv = abs(clauses_arr[cc, jj]) - 1
                            ss = 1 if clauses_arr[cc, jj] > 0 else 0
                            if vv == v_cand and assignment[vv] == ss:
                                brk += 1
                                break
                if brk < best_break:
                    best_break = brk
                    best_v = v_cand
            v = best_v
        # Flip variable and update clause_sat
        assignment[v] = 1 - assignment[v]
        for idx in range(var_offset[v], var_offset[v + 1]):
            cc = var_adj[idx]
            clause_sat[cc] = 0
            for jj in range(k):
                vv = abs(clauses_arr[cc, jj]) - 1
                ss = 1 if clauses_arr[cc, jj] > 0 else 0
                if assignment[vv] == ss:
                    clause_sat[cc] += 1

    return assignment, max_flips

# ── BSDT Gravity Engine ──────────────────────────────────────────────
class BSDTGravityEngine:
    def __init__(self, n, clauses, mu_scale=0.1,
                 G_max=0.05, top_k_frac=0.1, gravity_start=0.3):
        self.n = n
        self.clauses = clauses
        self.mu_scale = mu_scale
        self.G_max = G_max
        self.top_k_frac = top_k_frac
        self.gravity_start = gravity_start
        self.device = device
        # Pre-compute clause tensors
        m = len(clauses)
        self.vars_t = torch.zeros(m, 3, dtype=torch.long, device=device)
        self.signs_t = torch.zeros(m, 3, dtype=torch.float32, device=device)
        for i, (vs, ss) in enumerate(clauses):
            for j in range(3):
                self.vars_t[i, j] = vs[j]
                self.signs_t[i, j] = ss[j]

    def energy(self, s, mu):
        lit_vals = s[:, self.vars_t] * self.signs_t.unsqueeze(0)
        clause_energies = torch.prod(1.0 - lit_vals, dim=-1) / 8.0
        e_sat = clause_energies.sum(dim=-1)
        e_bin = mu * ((1.0 - s**2)**2).sum(dim=-1)
        return e_sat + e_bin

    def find_best_particle(self, s, clauses):
        with torch.no_grad():
            x = (s > 0).long()
            best_idx = 0
            best_sat = -1
            for i in range(s.shape[0]):
                _, ns = check_sat(x[i], clauses)
                if ns > best_sat:
                    best_sat = ns
                    best_idx = i
            return best_idx

    def gravity_flow(self, steps=4000, particles=1000, schedule='delay70', lr=0.02):
        s = torch.randn(particles, self.n, device=self.device) * 0.1
        s.requires_grad_(True)
        gravity_step = int(self.gravity_start * steps)
        top_k = max(1, int(self.top_k_frac * particles))

        for step in range(steps):
            # mu schedule
            if schedule == 'delay70':
                if step < int(0.7 * steps):
                    mu = 0.0
                else:
                    t_local = (step - int(0.7 * steps)) / (steps - int(0.7 * steps))
                    mu = self.mu_scale * 0.5 * (1 - np.cos(np.pi * t_local))
            else:
                t = step / steps
                mu = self.mu_scale * 0.5 * (1 - np.cos(np.pi * t))

            e = self.energy(s, mu)
            e_total = e.sum()
            e_total.backward()

            with torch.no_grad():
                s -= lr * s.grad
                # Gravity force after gravity_start
                if step >= gravity_step:
                    g_t = self.G_max * ((step - gravity_step) / (steps - gravity_step)) ** 2
                    energies = e.detach()
                    _, top_idx = energies.topk(top_k, largest=False)
                    center = s[top_idx].mean(dim=0, keepdim=True)
                    s += g_t * (center - s)
                s.clamp_(-1, 1)
            s.requires_grad_(True)
            if s.grad is not None:
                s.grad.zero_()

        return s.detach()

# ── Warmup Numba ──────────────────────────────────────────────────────
_ = walksat_numba(np.array([[1, -2, 3]], dtype=np.int32),
                  np.array([1, 0, 1], dtype=np.int32), max_flips=10)

print('✓ All dependencies + BSDTGravityEngine loaded')
print(f'  Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name()}')

In [ ]:
# ── H10 Cell 2: Gravity Experiment (hard α) + Charts ─────────────────
import matplotlib.pyplot as plt

alphas_hard = [3.8, 4.0, 4.2]
ns          = [500, 750, 1000]
N_INST      = 50
PARTICLES   = 1000
STEPS       = 4000

# H9b baseline (results without gravity)
baseline = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}

gravity_results = {}
print('=' * 80)
print('H10 GRAVITY ENGINE — hard α only')
print('=' * 80)
print(f"  {'α':>5} | {'n':>5} | {'S1':>6} | {'Final':>6} | {'Viols':>5} | "
      f"{'Flips':>7} | {'WS':>7} | {'Base':>5} | {'Δ':>6} | Time")
print('  ' + '-' * 74)

for alpha in alphas_hard:
    for n in ns:
        m = int(alpha * n)
        t0 = time.time()
        s1_ok = 0; ws_ok = 0; ws_tried = 0
        total_viols = 0; total_flips = 0; fail_count = 0

        for inst in range(N_INST):
            clauses = generate_3sat_instance(n, m)
            engine = BSDTGravityEngine(n, clauses, mu_scale=0.1,
                                       G_max=0.05, top_k_frac=0.1,
                                       gravity_start=0.3)
            s_final = engine.gravity_flow(steps=STEPS, particles=PARTICLES,
                                          schedule='delay70')
            # Check best particle
            best_idx = engine.find_best_particle(s_final, clauses)
            x_best = (s_final[best_idx] > 0).long()
            sat, n_sat = check_sat(x_best, clauses)

            if sat:
                s1_ok += 1
            else:
                viols = m - n_sat
                total_viols += viols
                fail_count += 1
                # Convert clauses for WalkSAT
                c_np = np.zeros((m, 3), dtype=np.int32)
                for ci2, (vs, ss) in enumerate(clauses):
                    for j2 in range(3):
                        c_np[ci2, j2] = (vs[j2] + 1) * ss[j2]
                x_np = x_best.cpu().numpy()
                sol, flips = walksat_numba(c_np, x_np.copy(),
                                          max_flips=100000, p=0.4)
                total_flips += flips
                ws_tried += 1
                if check_sat(torch.tensor(sol, device=device), clauses)[0]:
                    ws_ok += 1

        elapsed = time.time() - t0
        final_pct = (s1_ok + ws_ok) / N_INST * 100
        avg_viols = total_viols / max(fail_count, 1)
        avg_flips = total_flips // max(ws_tried, 1)
        base_pct = baseline[(alpha, n)]
        delta = final_pct - base_pct
        tag = '★' if final_pct >= 95 else ('▲' if delta > 0 else ('=' if delta == 0 else '▼'))

        gravity_results[(alpha, n)] = {
            's1': s1_ok / N_INST * 100, 'final': final_pct,
            'viols': avg_viols, 'flips': avg_flips,
            'ws': f'{ws_ok}/{ws_tried}', 'base': base_pct, 'delta': delta
        }
        print(f'  {alpha:5.1f} | {n:5d} | {s1_ok/N_INST*100:5.1f}% | {final_pct:5.1f}% | '
              f'{avg_viols:5.1f} | {avg_flips:7d} | {ws_ok:3d}/{ws_tried:<3d} | '
              f'{base_pct:4.0f}% | {delta:+5.1f}% | {elapsed:4.0f}s {tag}')
    print('  ' + '-' * 74)

print('\nDone! Generating charts...\n')

# ── Comparison Charts ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, n in enumerate(ns):
    ax = axes[i]
    base_vals = [baseline[(a, n)] for a in alphas_hard]
    grav_vals = [gravity_results[(a, n)]['final'] for a in alphas_hard]

    x_pos = range(len(alphas_hard))
    ax.bar([xi - 0.15 for xi in x_pos], base_vals, 0.3,
           label='H9b (no gravity)', color='#e74c3c', alpha=0.8)
    ax.bar([xi + 0.15 for xi in x_pos], grav_vals, 0.3,
           label='H10 (gravity)', color='#2ecc71', alpha=0.8)
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels([str(a) for a in alphas_hard])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n}')
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    # Delta annotations
    for xi, (b, g) in enumerate(zip(base_vals, grav_vals)):
        delta = g - b
        if delta != 0:
            ax.annotate(f'{delta:+.0f}%', (xi + 0.15, g + 2),
                        ha='center', fontsize=9, fontweight='bold',
                        color='green' if delta > 0 else 'red')

fig.suptitle('H10: Gravity Engine vs Baseline — Hard α Region',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('h10_gravity_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h10_gravity_vs_baseline.png')